# AI Production Studio — arabe standard vocalisé

Essai de **FastPitch Arabic + HiFi-GAN**, spécialisé en arabe standard moderne, avec le phonémiseur arabe de `nipponjo/tts_arabic`. Kokoro reste votre essai français séparé.

1. Exécutez les cellules dans l'ordre.
2. Lisez et corrigez le texte vocalisé avant la dernière cellule. La vocalisation automatique est désactivée : vos harakāt sont transmis au phonémiseur sans correction automatique.
3. Écoutez les voyelles, la shadda, le soukoun, le tanwin, les consonnes emphatiques et les fins de mots. Les règles de pause peuvent modifier la réalisation des désinences. Un test de tokens ne prouve pas une bonne prononciation.
4. Si l'essai convient, conservez le WAV et son JSON de provenance ; utilisez **Importer un média** dans le studio. Ce n'est pas un ZIP de tâche MMS.

Cet essai n'est pas un moteur de tajwid et ne garantit pas une diction parfaite. Il ne constitue pas une validation pédagogique ou linguistique. Le premier téléchargement des poids peut prendre du temps. Aucun abonnement ni clé API n'est ajouté. Calcul CPU ; une session GPU Colab connectée peut continuer à consommer ses unités.

Sources : [bibliothèque](https://github.com/nipponjo/tts_arabic), [modèles d'origine](https://github.com/nipponjo/tts-arabic-pytorch), [licences des composants](https://github.com/nipponjo/tts_arabic/blob/main/tts_arabic/ThirdPartyLicenses). Le dépôt contient des notices de tiers mais pas de licence racine complète à la révision choisie : cet essai n'établit pas une autorisation générale de redistribution des poids. Téléchargements depuis les liens Google Drive publiés par le mainteneur ; empreintes observées enregistrées pour reproduire l'essai, pas présentées comme des sommes officielles vérifiées.

Vérifié avant livraison : import CPU, distinctions de phonèmes, syntaxe des cellules. **Audio réel et compatibilité de cette session Colab à vérifier avec cet essai.**


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "onnxruntime==1.31.0", "gdown==5.2.0", "soundfile==0.13.1", "packaging==26.0"])
# Pas de modification de PyTorch, pas d'installation du moteur français.


In [ ]:
from pathlib import Path
import subprocess, sys

REVISION = "eae130c8b94ebf88d3fa727db63c740859106327"
source = Path("studio-arabic-source-" + REVISION[:12]).resolve()
if not source.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", "https://github.com/nipponjo/tts_arabic.git", str(source)])
actual = subprocess.check_output(["git", "-C", str(source), "remote", "get-url", "origin"], text=True).strip()
if actual != "https://github.com/nipponjo/tts_arabic.git":
    raise RuntimeError("Source inattendue : arrêt de l'essai.")
subprocess.check_call(["git", "-C", str(source), "checkout", "--detach", REVISION])
if subprocess.check_output(["git", "-C", str(source), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
    raise RuntimeError("Source modifiée : arrêt de l'essai.")
sys.path.insert(0, str(source))
print("Source épinglée. Les modèles seront téléchargés depuis les liens officiels du mainteneur.")


In [ ]:
"""Audition spécialisée MSA vocalisée. Dépendance nipponjo/tts_arabic épinglée dans le notebook."""
from pathlib import Path
import hashlib
import json
import re
import time

SOURCE_REVISION = "eae130c8b94ebf88d3fa727db63c740859106327"
SENTENCES = [
    "مَرْحَبًا بِكُمْ فِي وَرْشَتِنَا.",
    "عَلِمَ الطَّالِبُ. عُلِمَ الْخَبَرُ.",
    "دَرَسَ الطَّالِبُ. دَرَّسَ الْمُعَلِّمُ.",
    "هَذَا قَلَمٌ. كَتَبْتُ بِقَلَمٍ.",
]


def verify_phonemizer():
    from tts_arabic.text import arabic_to_tokens
    pairs = [("عَلِمَ", "عُلِمَ"), ("دَرَسَ", "دَرَّسَ"), ("قَلَمٌ", "قَلَمٍ")]
    evidence = []
    for left, right in pairs:
        a, b = arabic_to_tokens(left), arabic_to_tokens(right)
        if a == b:
            raise RuntimeError("Les signes ne sont pas distingués : " + left + " / " + right)
        evidence.append({"left": left, "right": right, "left_tokens": a, "right_tokens": b})
    return evidence


def generate(text, directory):
    if not isinstance(text, str) or not 1 <= len(text.strip()) <= 500:
        raise ValueError("Un texte de 1 à 500 caractères est nécessaire pour cet essai.")
    if not re.search(r"[\u064b-\u0652]", text):
        raise ValueError("Cet essai attend un texte arabe vocalisé, à vérifier avant synthèse.")
    folder = Path(directory)
    folder.mkdir(parents=True, exist_ok=True)
    output = folder / "essai-arabe.wav"
    if output.exists():
        raise FileExistsError("Changez de dossier pour conserver l'ancien essai.")
    from tts_arabic import tts
    import tts_arabic
    import numpy as np
    import soundfile as sf

    evidence = verify_phonemizer()
    started = time.perf_counter()
    samples = tts(text, speaker=0, pace=1.0, denoise=0.005, volume=0.9,
                  play=False, vowelizer=None, cuda=None,
                  model_id="fastpitch", vocoder_id="hifigan")
    elapsed = time.perf_counter() - started
    samples = np.asarray(samples)
    if not samples.size or not np.isfinite(samples).all() or not np.max(np.abs(samples)):
        raise RuntimeError("Le moteur n'a pas produit d'audio valide.")
    sf.write(str(output), samples, 22050, subtype="PCM_16")
    model_dir = Path(tts_arabic.__file__).parent / "data"
    hashes = {}
    for name in ["fp_ms.onnx", "hifigan.onnx", "denoiser.onnx"]:
        with (model_dir / name).open("rb") as stream:
            hashes[name] = hashlib.file_digest(stream, "sha256").hexdigest()
    report = {
        "source": "https://github.com/nipponjo/tts_arabic", "source_revision": SOURCE_REVISION,
        "model": "fastpitch", "vocoder": "hifigan", "speaker": 0,
        "language": "ar-MSA", "device": "CPU", "sample_rate": 22050,
        "text": text, "automatic_vowelizer": None, "pace": 1.0, "denoise": 0.005,
        "total_sec_including_first_download": round(elapsed, 3), "model_sha256_observed": hashes,
        "phonemizer_checks": evidence,
        "validation": "Signes distingués dans les tokens ; prononciation et naturel à contrôler à l'écoute.",
    }
    output.with_suffix(".json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return output


In [ ]:
from datetime import datetime, timezone
from IPython.display import Audio, display
from google.colab import files

TEXTE = "مَرْحَبًا بِكُمْ فِي وَرْشَتِنَا. عَلِمَ الطَّالِبُ. عُلِمَ الْخَبَرُ. دَرَسَ الطَّالِبُ. دَرَّسَ الْمُعَلِّمُ." #@param {type:"string"}
print("Texte transmis sans vocalisation automatique :", TEXTE)
print("Contrôles des voyelles, de la shadda et du tanwin :", verify_phonemizer())
dossier = "essai-arabe-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
audio_path = generate(TEXTE, dossier)
display(Audio(filename=str(audio_path)))
print("Écoutez et vérifiez la prononciation avant de retenir le moteur.")
files.download(str(audio_path))
files.download(str(audio_path.with_suffix(".json")))
